In [ ]:
# Images / representations with locality and translation invariance properties are the focus over here
# Images have structure :
# Locality → nearby pixels are correlated
# Stationarity → Same patterns occur at different locations
# Hierarchy    → Low level features combine into mid-level to high level features
# CNN's exploit this structure
#
#  Issue with MLP usage in images:
#  We are trying to learn a classification function
#  If an image has dimensions (224,224,3), on flattening it would be a 150,528-dim vector, which if we map to 128 dim to 2, then in first layer params only we get 19 mil
#  This is with the assumption any pixel could be related with any other pixel in arbitrary ways, only to learn 99% of parameters learning pixels are unrelated
#  
#  Instead of this CNN makes the assumptions of locality and translation invariance.
#  Extending this we use a filter (kernel) to look at small patches, instead of connecting every pixel to every hidden input 
#  And once we learn, say a filter for an edge we can use this to detect edges elsewhere as well, because of shared weights
#  This gives us, with 16 filters, 3*3*3*16 = 432 params
#  
#  The output for a (N,N) image with (k,k) kernel without padding or anything else would give (N-k+1,N-k+1) output (edge cases removed) 
#  And each component of this output for a particular filter tells us how much does the patch at this location in the original image match the filter's pattern
#  Output is a response map
#
#  Stacking layers:
#  eg) Layer 1 → 16 filters (3x3), each learns low-level patterns like edges in diff orientations, simple textures and output is 26x26x16
#  Layer 2 → 32 filters (3x3x16), each looks at all 16 feature maps from layer 1 to learn combination of low level features, output is 24x24x32
#  Layer 2 gives mid-level features similarly there could be a layer 3 for combinations of that to get high level objects, with filters of shape (3,3,32)
# The network discovers this hierarchy automatically through gradient descent on the classification loss.
#   
# Problem dependent architecture design
# Tradeoffs in CNN
# 1. Spatial resolution vs receptive field
# For a singular object to be meaningful in an image we need context → A single neuron in deeper layers needs large receptive fields to "see" a large region of the input image
# But to localise precisely high spatial resolution is needed, but you cant distinguish between pixels in the same feature map cell, they could be any pixel in the (n,n) cell obtained from previous layers
# So Large receptive field → many layers → feature maps shrink → low resolution → poor localization
# (A) You dont downsample → feature maps stay large → computational explosion (memory, compute)
# (B) You downsample → feature maps shrink → lose spatial precision
# 
# 2. Depth vs gradient flow
# To learn hierarchies (edges → textures → parts → objects), depth is needed
# More layers = more complex representations
# Deep networks learn better representations, but deep networks dont always train properly  - gradients vanish
#
# 3. Parameter efficiency vs Representation power
# The assumption of translation invariance makes CNN's more tractable, but limit what they can learn. like position dependent patterns
# For position specific processing, switch to either fully connected layers or attention mechanisms (compute which positions to attend to)
#
# Solutions and Costs
# 1. Pooling (controlled spatial downsampling)
# Reduce spatial dimension while retaining depth
# Types : Max,Mean,Sum,Min,Stochastic,L_p,Global Max/Average (at the end of fc)
# Max pooling is used mostly cause it tells directly if a feature exists without any dilution 
# Pooling costs spatial resolution, so useful in high-lvl classification and coarse localization, where pixel-lvl precision is not required.
# Not useful in segmentation masks, precise localisation, its a very lossy operation
#
# 2. Skip connections : Gradient highways
# If gradients can bypass some layers, they dont vanish
# ResNets (Residual Networks) use H(x) = F(x) + x , as the funcn to calc grad for, it adds 1 in the gradient.
# This provides a way for gradients to flow from layer 50 to 1 wihtout being multiplied by 50 jacobians
# Skip connections make training deep networks possible, not make depth necessary, if depth isnt needed skip connections are irrelevant
# 
# 3. Upsampling - recovering spatial resolution 
# Problem -> pooled to get a large receptive field, now  feature maps are 28×28, but you need 224×224 output
# You cant ever truly un-pool an image, get back lost information  without additional information (2nd law thermo)
# Naive methods : Nearest neighbors or Bilinear Interpolation - Fixed operations
# Nearest neighbor -> Each pixel in the 28x28 map becomes an 8x8 block in the 224x224 output
# Bilinear interpolation -> Smooth the transitions between values 
# Learned sampling : Transposed convolution
# Take a small input, produce a larger output by learning how to spread activations
# The kernel is learned, the network figures out how to upsample based on the task.
# Dont throw away information at all:
# U-net (U shaped encoder-decoder structure)
# Skip connections between Encoders (downsampling path) and Decoders (upsampling path)
# Decoders reuse the high res features present in the corresponding early layers of encoders
# More parameters (decoder channels double), More memory (store encoder feature maps)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,kernel_size=3,padding=1)   # 28×28×1 → 28×28×16
        self.pool  = nn.MaxPool2d(2,2)                         #stride = 2 so pooling cells are non-overlapping, 28×28 → 14×14
        self.conv2 = nn.Conv2d(16,32,padding=1,kernel_size=3)  # 14×14×16 → 14×14×32, pool again → 7×7×32
        self.conv3 = nn.Conv2d(32,64,kernel_size=3,padding=1)  # 7×7×32 → 7×7×64, pool again 3x3x64
        # self.conv4 = nn.Conv2d(64,128,kernel_size=3,padding=1) # 3x3x128 pool again 1x1x128
        self.fc    = nn.Linear(64*3*3,10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = self.pool(x)
        # x = torch.relu(self.conv4(x))
        # x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
    def fit(self,loader,epochs=5):

        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(self.parameters(), lr=0.01, momentum=0.9)
        self.train()
        for epoch in range(epochs):
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                loss = criterion(self(batch_x), batch_y)
                loss.backward()
                optimizer.step()
        return loss.item()
    
    def evaluate(self, loader):
        self.eval()
        criterion = nn.CrossEntropyLoss()
        total_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in loader:
                loss = criterion(self(batch_x), batch_y)
                total_loss += loss.item()
        
        return total_loss / len(loader)

In [ ]:
g = torch.Generator()
g.manual_seed(42)

transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data,batch_size=64,shuffle=True,generator=g)

test_loader = torch.utils.data.DataLoader(test_data,batch_size=64,shuffle=False)

model_SimpleCNN = SimpleCNN()
train_loss = model_SimpleCNN.fit(train_loader)
print(f"Final Training Loss: {train_loss}")

test_loss = model_SimpleCNN.evaluate(test_loader)
print(f"Average Testing Loss:{test_loss}")

In [ ]:
filters = model_SimpleCNN.conv1.weight.data.cpu().numpy()
fig,axes = plt.subplots(4,4,figsize=(10,10),)
for i,ax in enumerate(axes.flat):
    ax.imshow(filters[i,0,:,:])
    ax.set_title(f"Filter {i}")
    for row in range(3):
        for col in range(3):
            ax.text(col, row, f'{filters[i,0,row,col]:.2f}',ha='center',va='center',color='white',fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init

# class DeepCNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         layers = []
#         in_channels = 1
#         for i in range(10):
#             conv = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
#             layers.append(conv)
#             layers.append(nn.ReLU())
#             in_channels = 32
#             if i == 0: self.conv1 = conv
#             if i == 9: self.conv10 = conv

#         self.features = nn.Sequential(*layers)
#         self.fc = nn.Linear(32 * 28 * 28, 10)
    
#     def forward(self, x):
#         x = self.features(x)
#         x = x.view(x.size(0), -1)
#         x = self.fc(x)
#         return x

#     def fit(self, loader, epochs=5):
#         criterion = torch.nn.CrossEntropyLoss()
#         optimizer = torch.optim.SGD(self.parameters(), lr=0.01, momentum=0.9)
#         self.train()
#         for epoch in range(epochs):
#             for batch_idx,(batch_x, batch_y) in enumerate(loader):
#                 optimizer.zero_grad()
#                 loss = criterion(self(batch_x), batch_y)
#                 loss.backward()
#                 if batch_idx%100==0:
#                     norm1 = torch.norm(self.conv1.weight.grad).item()
#                     norm10 = torch.norm(self.conv10.weight.grad).item()
#                     print(f"Conv1 Grad Norm: {norm1} | Conv10 Grad Norm: {norm10}")

#                 optimizer.step()
#         return loss.item()
    
#     def evaluate(self, loader):
#         self.eval()
#         criterion = nn.CrossEntropyLoss()
#         total_loss = 0
#         with torch.no_grad():
#             for batch_x, batch_y in loader:
#                 loss = criterion(self(batch_x), batch_y)
#                 total_loss += loss.item()
        
#         return total_loss / len(loader)
class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        in_channels = 1
        for i in range(10):
            conv = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm2d(32))
            in_channels = 32
            if i == 0: self.conv1 = conv
            if i == 9: self.conv10 = conv

        self.fc = nn.Linear(32 * 28 * 28, 10)
        
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        for i in range(len(self.convs)):
            identity = x
            x = self.convs[i](x)
            x = self.bns[i](x)
            x = torch.relu(x)
            if i > 0:
                x = x + identity 
        
        x = x.view(x.size(0), -1)
        return self.fc(x)

    def fit(self, loader, epochs=5):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(self.parameters(), lr=0.001, momentum=0.9)
        self.train()
        for epoch in range(epochs):
            for batch_idx,(batch_x, batch_y) in enumerate(loader):
                optimizer.zero_grad()
                loss = criterion(self(batch_x), batch_y)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                
                if batch_idx%1000==0:
                    norm1 = torch.norm(self.conv1.weight.grad).item()
                    norm10 = torch.norm(self.conv10.weight.grad).item()
                    print(f"Conv1 Grad Norm: {norm1} | Conv10 Grad Norm: {norm10}")

                optimizer.step()
        return loss.item()
    
    def evaluate(self, loader):
        self.eval()
        criterion = nn.CrossEntropyLoss()
        total_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in loader:
                total_loss += criterion(self(batch_x), batch_y).item()
        return total_loss / len(loader)

In [ ]:
g = torch.Generator()
g.manual_seed(42)

transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data,batch_size=64,shuffle=True,generator=g)

test_loader = torch.utils.data.DataLoader(test_data,batch_size=64,shuffle=False)

model_DeepCNN = DeepCNN()
train_loss = model_DeepCNN.fit(train_loader)
print(f"Final Training Loss: {train_loss}")

test_loss = model_DeepCNN.evaluate(test_loader)
print(f"Average Testing Loss:{test_loss}")